# makeshift — quick start

The core workflow: fetch a BMRB entry, get tidy chemical shifts, simulate an HSQC, handle multi-entity entries, and compute the Chemical Shift Index (CSI).

In [ ]:
import makeshift as ms
import seaborn as sns
import matplotlib.pyplot as plt

## 1. Fetch a BMRB entry

`ChemicalShifts.from_bmrb` downloads the NMR-STAR file and parses it into a tidy dataframe — one row per chemical shift, in `cs.data`.

In [ ]:
# Unphosphorylated NtrCr (Volkman & Kern 2001)
cs = ms.ChemicalShifts.from_bmrb(4527)
cs.data.head()

Key columns: `Seq_ID` (residue number), `Comp_ID` (3-letter residue), `Atom_ID`, `Atom_type` (element), `Val` (shift, ppm), `Entity_ID`.

## 2. Simulate an HSQC

Pivot to one row per residue with N and H columns.

In [ ]:
hsqc = (
    cs.data[cs.data['Atom_ID'].isin(['N', 'H'])]
    .pivot_table(index=['Seq_ID', 'Comp_ID'], columns='Atom_ID', values='Val')
    .reset_index()
)

sns.scatterplot(x='H', y='N', data=hsqc)
plt.gca().invert_xaxis(); plt.gca().invert_yaxis()
plt.xlabel('¹H (ppm)'); plt.ylabel('¹⁵N (ppm)')
plt.title('Simulated HSQC — NtrCr (4527)')

## 3. Multi-entity entries

Some entries hold several entities (protein + DNA + ligand). `sequences()` lists them.

In [ ]:
# hERR2 zinc finger bound to DNA
cs = ms.ChemicalShifts.from_bmrb(5363)
cs.sequences()

In [ ]:
# keep only the polypeptide entity
ents = cs.sequences()
protein_ids = ents.loc[ents['Polymer_type'] == 'polypeptide(L)', 'ID']
cs.data[cs.data['Entity_ID'].isin(protein_ids)].head()

## 4. Chemical Shift Index (CSI)

`calc_csi=True` adds `csi_raw` (the (CA−CB) secondary shift: positive = helix, negative = strand) and `csi` (discretised to ±1 / 0).

In [ ]:
# KaiB-TV4 — known secondary structure
cs = ms.ChemicalShifts.from_bmrb(31107, calc_csi=True)

seq = 'MYVFRLYVRGETHAAEVALKNLHDLLSSALKVPYTLKVVDVTKQPDLAEKDQVQATPTLVRVYPQPVRRLVGQLDHRYRLQHLLSP'
dss = 'CEEEEEEECCCHHHHHHHHHHHHHHHHHHHCCCCEEEEEECCCHHHHHHHHHCCCCCEEEECCCCCCEEEECCCCHHHHHHHHHHC'

tmp = cs.data[cs.data['Atom_type'] == 'N'].copy()
tmp['dssp'] = tmp['Seq_ID'].map(lambda i: dss[i - 1])

plt.figure(figsize=(10, 2))
sns.scatterplot(x='Seq_ID', y='csi_raw', hue='dssp', data=tmp)
plt.axhline(0, color='k', lw=0.5, ls='--')
plt.xlabel('Residue'); plt.ylabel('CSI (ppm)')
plt.title('CSI — KaiB-TV4 (31107)'); plt.legend(bbox_to_anchor=(1, 1))